# Generative models as unsupervised learning

**Accompanies Section 10 of** *Best Practices for Unsupervised Learning in Molecular Systems* (Article v1.0).

This notebook is scoped narrowly, because learning a distribution over molecules from
unlabeled structures is unambiguously unsupervised, while conditional and goal-directed
generation (generating molecules to hit a property profile) is driven by supervised predictors
and oracles and belongs to a different guide.

So we treat a variational autoencoder as what it is from this article's point of view: **a learned
dimensionality reduction whose latent space must be validated like any other embedding**.
The latent space is the compressed representation the encoder learns, one vector per molecule.

### Learning objectives
- Train a small SMILES VAE on a laptop
- Diagnose prior/posterior mismatch, the failure that quietly turns a VAE into a plain autoencoder
- Apply the embedding diagnostics from Section 7 to a latent space
- Evaluate generated molecules beyond validity

### What this notebook is designed to make go wrong
A latent space that looks beautifully organized by a property, where the organization is entirely
explained by molecular size.

### What you need installed
PyTorch (`pip install torch`), which is the only optional dependency in the repository and is needed by this notebook alone. RDKit handles the molecules, and ASE reads the QM7 structure file used by the manuscript figure at the end.

### Roughly how long it takes
Around ten to fifteen minutes on a laptop CPU, nearly all of it training the autoencoder, which prints only once an epoch finishes.

In [ ]:
import os

# PyTorch (imported below) ships its own OpenMP runtime and conda's numpy links
# another; with two OpenMP runtimes in one process the training loop can deadlock
# on macOS rather than merely run slowly. Pin the process to a single thread and
# allow the duplicate runtime. These have to be set before numpy is imported,
# because the OpenMP libraries read them once, at load time.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# One palette for every figure here. The series stay distinguishable in
# grayscale as well as in color, and marker and dash vary alongside the color,
# so nothing depends on color alone.
TEAL, PURPLE, LAVENDER, GREEN, PLUM, SLATE = (
    "#2D4F54", "#7B539E", "#B8A0D2", "#5A9448", "#9E4A78", "#3A3D4A"
)
PALETTE = [TEAL, PURPLE, LAVENDER, GREEN]


def set_style():
    """Apply the plot style used throughout these notebooks."""
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "axes.edgecolor": SLATE,
        "axes.labelcolor": SLATE,
        "axes.titlecolor": SLATE,
        "axes.linewidth": 1.0,
        "axes.grid": False,
        "xtick.color": SLATE,
        "ytick.color": SLATE,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "legend.frameon": False,
        "axes.prop_cycle": (
            mpl.cycler(color=PALETTE)
            + mpl.cycler(marker=["o", "s", "^", "D"])
            + mpl.cycler(linestyle=["-", (0, (4, 1.5)), (0, (1, 1.2)), (0, (5, 1.2, 1, 1.2))])
        ),
    })


set_style()
warnings.filterwarnings("ignore", category=FutureWarning)

# Fix a seed so the notebook reproduces. That is not the same as checking a
# conclusion survives a different seed, which we do explicitly where it matters.
SEED = 20260726
rng = np.random.default_rng(SEED)

# The example data, fetched by scripts/download_data.py.
DATA = Path.cwd().parent / "data"

# PyTorch is the only optional dependency here. Say so now, instead of failing
# with a bare ImportError somewhere in the middle of the notebook.
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    torch.manual_seed(SEED)
    # Single-threaded for the same reason as the env vars above: it keeps the
    # CPU training loop from deadlocking on the OpenMP runtime on macOS.
    torch.set_num_threads(1)
except ImportError:
    print("PyTorch is missing. Install it with: pip install torch")

DEVICE = "cpu"     # everything here is sized to run on a laptop CPU

## 1. Standardize and split the data before anything is fitted

### Standardize every structure to one convention

Standardization brings every structure to one documented convention before a single descriptor is
computed. Skip it and two records describing the same compound can sit in different regions of
descriptor space, land in different clusters, and get reported as a chemical distinction.

The function below is the short version of Section 4: parse and sanitize, keep the largest
fragment so that salts and counter-ions go, neutralize formal charges, and write a canonical
SMILES. Tautomer canonicalization and stereochemistry removal are left out. Both are chemical
judgments and this data set does not need either, but say in your methods which steps you ran and
which you skipped.

Build the RDKit helper objects once, outside the function. Constructing them per molecule
dominates the runtime on a set of any size.

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")      # parse failures are counted, not printed

LARGEST_FRAGMENT = rdMolStandardize.LargestFragmentChooser()
UNCHARGER = rdMolStandardize.Uncharger()


def standardize(smiles):
    """One SMILES brought to a single convention, or None if it will not parse."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    if "." in smiles:
        mol = LARGEST_FRAGMENT.choose(mol)
    mol = UNCHARGER.uncharge(mol)
    Chem.SanitizeMol(mol)
    return Chem.MolToSmiles(mol, canonical=True)

### Split by scaffold, not at random

Molecules sharing a Bemis-Murcko scaffold are near-neighbors of each other. Split them at random
and the held-out set fills up with analogues of the training set, so the "novel" molecules we
generate later would be rediscoveries. Grouping by scaffold keeps a whole scaffold on one side of
the split.

One detail in the code carries most of the weight. Groups are visited largest first, and a group
joins the test set only when doing so brings the achieved fraction *closer* to the target. Adding
groups while below the target overshoots badly when one scaffold covers a large slice of the
library, and congeneric series routinely do exactly that.

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold


def scaffold_split(smiles, test_size=0.2, random_state=0):
    """Split so that no Bemis-Murcko scaffold appears on both sides."""
    scaffolds = []
    for i, smi in enumerate(smiles):
        core = MurckoScaffold.GetScaffoldForMol(Chem.MolFromSmiles(smi))
        key = Chem.MolToSmiles(core)
        # An acyclic molecule has an empty scaffold. Give each of those its own
        # group; pooling them would make the pool leak across the split.
        scaffolds.append(key if key else f"__acyclic_{i}__")

    groups = np.asarray(scaffolds)
    rng = np.random.default_rng(random_state)
    unique, counts = np.unique(groups, return_counts=True)
    # Largest group first, with a small jitter so that ties break reproducibly.
    order = np.argsort(-counts + rng.uniform(0, 0.5, size=counts.shape))

    target = test_size * groups.size
    test_groups, n_in_test = [], 0
    for idx in order:
        if abs(n_in_test + counts[idx] - target) < abs(n_in_test - target):
            test_groups.append(unique[idx])
            n_in_test += counts[idx]

    achieved = n_in_test / groups.size
    if abs(achieved - test_size) > 0.05:
        print(f"  note: the group sizes only permit a test fraction of {achieved:.2f}. "
              "Report the achieved fraction, not the requested one.")

    mask = np.isin(groups, test_groups)
    return np.where(~mask)[0], np.where(mask)[0]

In [ ]:
# A few thousand molecules, enough for a small SMILES VAE to learn valid grammar
# and still finish on a laptop CPU. The autoregressive decoder below needs more
# than a few hundred examples before its samples start to parse.
zinc = pd.read_csv(DATA / "zinc-250k-sample.csv").head(2500)
smiles = sorted({s for s in (standardize(x.strip()) for x in zinc["smiles"])
                 if s is not None})
smiles = [s for s in smiles if len(s) <= 72]
print(f"{len(smiles)} molecules after standardization and length filtering")

# Scaffold split, so that the 'novel' molecules we generate later are not
# rediscoveries of held-out analogues of the training set.
train_idx, test_idx = scaffold_split(smiles, test_size=0.15, random_state=SEED)
train_smiles = [smiles[i] for i in train_idx]
test_smiles = [smiles[i] for i in test_idx]
print(f"train {len(train_smiles)}  test {len(test_smiles)}")

In [ ]:
from rdkit.Chem import Draw
from IPython.display import display


def draw_molecules(smiles_list, legends=None, n=8, per_row=4):
    """Grid depiction of the first n molecules, for orientation rather than analysis.

    Molecules that fail to parse are dropped along with their legend, so a single
    bad string does not blank the whole grid.
    """
    mols = [Chem.MolFromSmiles(s) for s in smiles_list[:n]]
    legs = list(legends[:n]) if legends is not None else None
    keep = [i for i, m in enumerate(mols) if m is not None]
    mols = [mols[i] for i in keep]
    legs = [legs[i] for i in keep] if legs is not None else None
    return Draw.MolsToGridImage(mols, molsPerRow=per_row, subImgSize=(260, 200), legends=legs)


# The molecules the VAE will learn from, so the samples it produces later have
# something to be compared against by eye.
display(draw_molecules(train_smiles, n=8))

In [ ]:
# A character vocabulary with three special tokens: a start token the decoder is
# primed with, an end token it learns to emit, and a pad token that fills the
# rest of each fixed-width row and is ignored by the loss.
BOS, EOS, PAD = "^", "$", " "
charset = [PAD, BOS, EOS] + sorted({c for s in smiles for c in s})
char_to_idx = {c: i for i, c in enumerate(charset)}
VOCAB = len(charset)
PAD_IDX, BOS_IDX, EOS_IDX = char_to_idx[PAD], char_to_idx[BOS], char_to_idx[EOS]
MAX_LEN = max(len(s) for s in smiles) + 2      # room for the start and end tokens


def encode(smiles_list):
    """Token-id matrix: a leading start token, the characters, an end token, then padding."""
    out = np.full((len(smiles_list), MAX_LEN), PAD_IDX, dtype=np.int64)
    for i, s in enumerate(smiles_list):
        toks = [BOS_IDX] + [char_to_idx[c] for c in s] + [EOS_IDX]
        out[i, :len(toks)] = toks
    return torch.tensor(out)


X_train = encode(train_smiles)
X_test = encode(test_smiles)
print(f"vocabulary {VOCAB} (incl. start/end/pad), max length {MAX_LEN}")
print(f"tensors: {tuple(X_train.shape)}, {tuple(X_test.shape)}")

## 2. Build the model and get both loss terms right

The loss is two terms added together: a reconstruction term, and a KL term that pulls the
encoded distribution toward the simple N(0, I) prior. When the KL term dominates, the latent
stops encoding anything about the molecule and the decoder falls back on the average of the
training set. That failure is called posterior collapse, and we hold it off with a KL warm-up
and a small free-bits floor on each latent dimension.

Three details matter, and all three are commonly done wrong.

**The decoder must be autoregressive.** At each position it sees the previous character together
with the latent, and during training it sees the *true* previous character, which is called
teacher forcing. A decoder that instead has to emit an entire SMILES string from the latent alone
never learns the grammar, and essentially nothing it produces parses as a molecule. This one
change is what takes generated validity from zero to a large fraction of samples.

**The reconstruction loss must match the output.** The decoder produces a categorical
distribution over the vocabulary at each position, so the loss is categorical cross-entropy.
Binary cross-entropy would treat the vocabulary entries as independent Bernoulli variables,
which is inconsistent with a softmax output whose entries sum to one.

**The reparameterization must not be rescaled.** The standard trick is `z = mu + sigma * eps`
with `eps ~ N(0, 1)`. Multiplying `eps` by a small constant (which is easy to do by accident)
shrinks the injected noise relative to what the KL term regularizes against, and the model
trains as a near-deterministic autoencoder while appearing to be a VAE.

In [ ]:
class MolecularVAE(nn.Module):
    def __init__(self, vocab, max_len, latent=64, hidden=256, emb=48, layers=2):
        super().__init__()
        self.vocab, self.max_len, self.latent = vocab, max_len, latent
        self.hidden, self.layers = hidden, layers
        self.enc = nn.Sequential(
            nn.Conv1d(vocab, 32, 9), nn.ReLU(),
            nn.Conv1d(32, 32, 9), nn.ReLU(),
            nn.Conv1d(32, 48, 11), nn.ReLU(), nn.Flatten(),
        )
        with torch.no_grad():
            flat = self.enc(torch.zeros(1, vocab, max_len)).shape[1]
        self.fc_mu = nn.Linear(flat, latent)
        self.fc_logvar = nn.Linear(flat, latent)
        # Decoder: an embedding for the previous character, a GRU whose initial
        # state is set from the latent, and the latent supplied again as context
        # at every step.
        self.embed = nn.Embedding(vocab, emb)
        self.z_to_h = nn.Linear(latent, layers * hidden)
        self.gru = nn.GRU(emb + latent, hidden, layers, batch_first=True)
        self.out = nn.Linear(hidden, vocab)

    def encode(self, x_ids):
        onehot = F.one_hot(x_ids, self.vocab).float()
        h = self.enc(onehot.transpose(1, 2))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        # eps ~ N(0, 1). NOT scaled by a small constant; see the markdown above.
        eps = torch.randn_like(logvar)
        return mu + torch.exp(0.5 * logvar) * eps

    def decode(self, z, inp_ids):
        """Teacher-forced decode: predict each character from the previous one and z."""
        h0 = torch.tanh(self.z_to_h(z)).view(-1, self.layers, self.hidden)
        h0 = h0.transpose(0, 1).contiguous()
        e = self.embed(inp_ids)
        context = z.unsqueeze(1).expand(-1, e.size(1), -1)
        out, _ = self.gru(torch.cat([e, context], dim=-1), h0)
        return self.out(out)                     # logits, not probabilities

    def forward(self, x_ids):
        mu, logvar = self.encode(x_ids)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z, x_ids[:, :-1])   # predict token t+1 from token t
        return logits, mu, logvar


def vae_loss(logits, target_ids, mu, logvar, beta=1.0, free_bits=0.15):
    # Categorical cross-entropy over the vocabulary, ignoring pad positions.
    recon = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target_ids.reshape(-1),
        ignore_index=PAD_IDX,
        reduction="sum",
    )
    # Free bits: clamp each latent dimension's KL at a floor, so the optimiser
    # cannot lower the loss by switching a dimension off (posterior collapse).
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    kl = torch.clamp(kl_per_dim, min=free_bits).sum()
    return recon + beta * kl, recon, kl


model = MolecularVAE(VOCAB, MAX_LEN).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 40
loader = DataLoader(TensorDataset(X_train), batch_size=128, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = []
for epoch in range(EPOCHS):
    model.train()
    totals = np.zeros(3)
    for (batch,) in loader:
        optimizer.zero_grad()
        logits, mu, logvar = model(batch)
        # KL warm-up: ramp beta from 0, which avoids posterior collapse early on.
        beta = min(1.0, (epoch + 1) / 15)
        # The decoder predicts token t+1, so the target is the input shifted by one.
        loss, recon, kl = vae_loss(logits, batch[:, 1:], mu, logvar, beta=beta)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        totals += [loss.item(), recon.item(), kl.item()]
    scheduler.step()

    model.eval()
    with torch.no_grad():
        logits, mu, logvar = model(X_test)
        val_loss = vae_loss(logits, X_test[:, 1:], mu, logvar)[0].item() / len(X_test)
    history.append((totals[0] / len(X_train), val_loss))
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}  train {history[-1][0]:9.2f}  val {val_loss:9.2f}  "
              f"lr {scheduler.get_last_lr()[0]:.2e}")

In [ ]:
history = np.array(history)
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.plot(history[:, 0], label="train")
ax.plot(history[:, 1], label="validation")
ax.set_xlabel("epoch"); ax.set_ylabel("loss per molecule")
ax.set_title("Training curve"); ax.legend(frameon=False)
plt.show()

## 3. Check whether the posterior matches the prior

If the aggregate posterior has drifted away from the N(0, I) prior, sampling from the prior will
produce poor molecules while sampling near encoded training points looks fine, and it is easy to
report only the second and conclude the model works.

The cell below also counts active latent dimensions, the ones whose encoded values still vary
from one molecule to the next. A count near zero is how posterior collapse shows up.

In [ ]:
model.eval()
with torch.no_grad():
    mu_train, logvar_train = model.encode(X_train)

mu_np = mu_train.numpy()
print("Aggregate posterior vs the N(0, I) prior it is regularized toward:")
print(f"  mean of mu   : {mu_np.mean():+.3f}   (prior: 0.000)")
print(f"  std  of mu   : {mu_np.std():.3f}    (prior: 1.000)")
print(f"  mean sigma   : {np.exp(0.5 * logvar_train.numpy()).mean():.3f}")

active = np.sum(mu_np.std(axis=0) > 0.1)
print(f"\n  active latent dimensions: {active}/{model.latent}")
if active < model.latent * 0.5:
    print("  WARNING: most latent dimensions have collapsed; the model is using")
    print("  only a fraction of its latent space and is close to a deterministic AE.")

## 4. The latent space is an embedding, so validate it as one

Everything from Section 7 applies here. In particular, coloring a latent space by a property
and seeing structure is weak evidence, because a property correlated with molecular size will
structure almost any embedding.

In [ ]:
from rdkit.Chem.Descriptors import MolLogP, MolWt
from sklearn.decomposition import PCA

latent_2d = PCA(n_components=2, random_state=SEED).fit_transform(mu_np)
train_mols = [Chem.MolFromSmiles(s) for s in train_smiles]
weights = np.array([MolWt(m) for m in train_mols])
logp = np.array([MolLogP(m) for m in train_mols])
lengths = np.array([len(s) for s in train_smiles])

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, color_by, name in [(axes[0], lengths, "SMILES length"),
                         (axes[1], weights, "molecular weight"),
                         (axes[2], logp, "logP")]:
    sc = ax.scatter(latent_2d[:, 0], latent_2d[:, 1], c=color_by, s=6, cmap="viridis", linewidths=0)
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(sc, ax=ax, fraction=0.045)
plt.tight_layout(); plt.show()

print("Correlation of latent PC1 with:")
for name, values in [("SMILES length", lengths), ("molecular weight", weights), ("logP", logp)]:
    print(f"  {name:18s} {abs(np.corrcoef(latent_2d[:, 0], values)[0, 1]):.3f}")
print()
print("If length and weight dominate, the 'chemical organization' of this latent")
print("space is largely an organization by size. Check that before you interpret")
print("any cluster found in it.")

### Score the latent space against a random projection baseline

Three numbers do the work, all of them from Section 7. Trustworthiness asks whether the
embedding invented neighbors: points that look close in 2D but were far apart in the original
space. Continuity is its dual and asks which real neighbors it lost; it is trustworthiness
computed with the two spaces exchanged, which is the standard identity. Neighbor preservation
is the most directly interpretable of the three, the fraction of each point's k nearest
neighbors that survived the reduction.

The random projection baseline goes in by default and you should keep it. It costs nothing, it
comes with distortion guarantees (Johnson-Lindenstrauss), and it calibrates how much of an
embedding's apparent quality comes from the method and how much from the data being easy.

In [ ]:
from sklearn.manifold import trustworthiness
from sklearn.neighbors import NearestNeighbors
from sklearn.random_projection import GaussianRandomProjection


def neighbor_preservation(X, embedding, n_neighbors=12):
    """Mean fraction of each point's k nearest neighbors kept by the embedding."""
    def knn(A):
        finder = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(A)
        return finder.kneighbors(A, return_distance=False)[:, 1:]

    original, embedded = knn(X), knn(embedding)
    return float(np.mean([np.intersect1d(original[i], embedded[i]).size / n_neighbors
                          for i in range(len(original))]))


def evaluate_embeddings(X, embeddings, n_neighbors=12, random_state=0):
    """Score each embedding of X against a random projection baseline."""
    embeddings = dict(embeddings)
    n_components = next(iter(embeddings.values())).shape[1]
    embeddings["random projection"] = GaussianRandomProjection(
        n_components=n_components, random_state=random_state
    ).fit_transform(X)

    rows = [
        {
            "method": name,
            "trustworthiness": float(trustworthiness(X, emb, n_neighbors=n_neighbors)),
            # Continuity is trustworthiness with the two spaces exchanged.
            "continuity": float(trustworthiness(emb, X, n_neighbors=n_neighbors)),
            f"{n_neighbors}nn_preserved": neighbor_preservation(X, emb, n_neighbors),
        }
        for name, emb in embeddings.items()
    ]
    return (pd.DataFrame(rows)
            .sort_values("trustworthiness", ascending=False)
            .reset_index(drop=True))

In [ ]:
quality = evaluate_embeddings(mu_np, {"latent PCA": latent_2d}, n_neighbors=12, random_state=SEED)
print(quality.round(3).to_string(index=False))

Here is how to read that table. These are the same embedding diagnostics used in notebook 07, now applied to a latent space, because a latent space is an embedding and gets judged like one. Trustworthiness and continuity run from 0 to 1 and trade off against each other, so read them as a pair instead of picking the flattering one.

There is no threshold at which a latent space becomes acceptable. What makes these numbers mean something is the comparison next to them: PCA on the same input, fitted in a second, with no training and no hyperparameters. A latent space that does not beat it has not earned the complexity, and reporting the autoencoder's numbers alone would hide that. Here trustworthiness and continuity both sit near 1.0 for every method, because the latent has only a few dominant directions and two PCA components already capture almost all of them; the neighbor-preservation column is the one that still separates the latent PCA from the random-projection floor, so it is the column to read here.

### Walk a straight line between two molecules

The metrics above are a proxy for a question they never quite ask: is the latent space smooth enough that points between two molecules decode to something sensible? Encode two test molecules, take evenly spaced points along the straight line joining their latent means, and decode each. A space that has learned a distribution gives a path of valid structures that change gradually; a space that has only memorized training points gives invalid strings in between. Read the fraction of intermediate points that parse rather than predicting it, and expect a small laptop model to leave gaps. Sampling from the prior and the reconstruction-versus-prior gap, the other half of the usual latent check, are already reported in the validity, uniqueness and novelty section that follows.

In [ ]:
@torch.no_grad()
def decode_greedy(z):
    """Greedy autoregressive decode of latent vectors to SMILES.

    A deterministic path is what interpolation wants. This mirrors the sampler
    defined in the next section; it is repeated here so this cell stands alone.
    """
    b = z.size(0)
    h = torch.tanh(model.z_to_h(z)).view(b, model.layers, model.hidden).transpose(0, 1).contiguous()
    token = torch.full((b, 1), BOS_IDX, dtype=torch.long)
    done = torch.zeros(b, dtype=torch.bool)
    rows = []
    for _ in range(MAX_LEN - 1):
        e = model.embed(token)
        out, h = model.gru(torch.cat([e, z.unsqueeze(1)], dim=-1), h)
        nxt = model.out(out[:, -1]).argmax(-1)
        nxt = torch.where(done, torch.full_like(nxt, PAD_IDX), nxt)
        rows.append(nxt)
        done = done | (nxt == EOS_IDX)
        token = nxt.unsqueeze(1)
        if done.all():
            break
    ids = torch.stack(rows, dim=1).numpy()
    out = []
    for r in ids:
        ch = []
        for t in r:
            if t == EOS_IDX:
                break
            if t not in (PAD_IDX, BOS_IDX):
                ch.append(charset[t])
        out.append("".join(ch))
    return out


# Decoding steps the GRU one token at a time. Cap the thread count for that
# loop: it is inherently sequential, so one thread is instant here and avoids
# the CPU thread oversubscription that can otherwise stall a long-lived kernel.
torch.set_num_threads(1)

with torch.no_grad():
    mu_ends, _ = model.encode(X_test[[0, 1]])
alphas = np.linspace(0.0, 1.0, 10)
z_path = torch.stack([(1 - a) * mu_ends[0] + a * mu_ends[1] for a in alphas])
path = decode_greedy(z_path)

print(f"endpoint A: {test_smiles[0]}")
print(f"endpoint B: {test_smiles[1]}\n")
n_valid = 0
for a, s in zip(alphas, path):
    # An empty decode parses to an empty RDKit molecule rather than to None, so
    # it would count as valid; require a non-empty string as well.
    ok = bool(s) and Chem.MolFromSmiles(s) is not None
    n_valid += ok
    print(f"  alpha = {a:.2f}   {'valid  ' if ok else 'invalid'}   {s}")
print(f"\n{n_valid}/{len(alphas)} points along the interpolation parse as molecules")

## 5. Report uniqueness and novelty alongside validity

Of the molecules a model generates, validity is the fraction that parse as chemistry at all,
uniqueness the fraction distinct from one another, and novelty the fraction absent from the
reference set. All three are necessary and insufficient: a model that emits one valid molecule
over and over scores perfectly on validity and tells you nothing. Report them together, expect
trade-offs, and state the reference set used for novelty, because it changes the number materially.

We first check that the decoder learned to produce valid SMILES at all, by reconstructing training
molecules from their own encodings, and then sample two ways: near encoded training molecules, and
from the prior. Sampling uses a temperature below one, which trades a little validity for the
diversity that makes uniqueness meaningful; greedy decoding maximizes validity but collapses onto a
handful of molecules, so its uniqueness is near zero and its V.U.N. row is uninformative.

In [ ]:
@torch.no_grad()
def sample(z, temperature=0.6, greedy=False):
    """Autoregressively decode one SMILES per row of z, priming with the start token."""
    batch = z.size(0)
    h = torch.tanh(model.z_to_h(z)).view(batch, model.layers, model.hidden)
    h = h.transpose(0, 1).contiguous()
    token = torch.full((batch, 1), BOS_IDX, dtype=torch.long)
    finished = torch.zeros(batch, dtype=torch.bool)
    rows = []
    for _ in range(MAX_LEN - 1):
        e = model.embed(token)
        out, h = model.gru(torch.cat([e, z.unsqueeze(1)], dim=-1), h)
        logits = model.out(out[:, -1])
        if greedy:
            nxt = logits.argmax(-1)
        else:
            nxt = torch.multinomial(F.softmax(logits / temperature, dim=-1), 1).squeeze(-1)
        nxt = torch.where(finished, torch.full_like(nxt, PAD_IDX), nxt)
        rows.append(nxt)
        finished = finished | (nxt == EOS_IDX)
        token = nxt.unsqueeze(1)
        if finished.all():
            break
    ids = torch.stack(rows, dim=1).numpy()
    out = []
    for row in ids:
        chars = []
        for t in row:
            if t == EOS_IDX:
                break
            if t not in (PAD_IDX, BOS_IDX):
                chars.append(charset[t])
        out.append("".join(chars))
    return out


def evaluate(generated, reference):
    parsed = [Chem.MolFromSmiles(s) for s in generated if s]
    valid = [m for m in parsed if m is not None]
    # Bring each generated molecule to the same convention the reference set was
    # built with before asking whether it is novel. Without this a molecule that
    # differs from a training one only by protonation or a stray counterion would
    # be counted as new, inflating novelty. standardize() returns None only for
    # strings that will not parse, which these already have.
    keys = [k for k in (standardize(Chem.MolToSmiles(m)) for m in valid) if k]
    unique = set(keys)
    novel = unique - set(reference)
    return {
        "validity": len(valid) / len(generated) if generated else 0.0,
        "uniqueness": len(unique) / len(keys) if keys else 0.0,
        "novelty": len(novel) / len(unique) if unique else 0.0,
    }


# Decoding steps the GRU one token at a time. Cap the thread count for that
# loop: it is inherently sequential, so one thread is instant here and avoids
# the CPU thread oversubscription that can otherwise stall a long-lived kernel.
torch.set_num_threads(1)
N, TEMPERATURE = 400, 0.6
reference = set(train_smiles)
with torch.no_grad():
    picks = torch.tensor(rng.choice(len(mu_train), N))
    # a check that the decoder learned valid SMILES at all: greedily reconstruct
    # training molecules from their own encodings
    recon_validity = evaluate(sample(mu_train[picks], greedy=True), reference)["validity"]
    # (a) sample near encoded training molecules
    near_data = sample(mu_train[picks] + 0.1 * torch.randn(N, model.latent), temperature=TEMPERATURE)
    # (b) sample the prior: what the model claims to have learned
    from_prior = sample(torch.randn(N, model.latent), temperature=TEMPERATURE)

print(f"greedy reconstruction of training molecules: {recon_validity:.0%} valid\n")
print(f"sampling at temperature {TEMPERATURE}")
print(f"{'sampling scheme':<28s} {'validity':>9s} {'uniqueness':>11s} {'novelty':>9s}")
results = {}
for name, gen in [("near encoded molecules", near_data), ("from the prior", from_prior)]:
    m = evaluate(gen, reference)
    results[name] = m
    print(f"{name:<28s} {m['validity']:9.3f} {m['uniqueness']:11.3f} {m['novelty']:9.3f}")

print()
print("Both schemes produce valid, unique, novel molecules, and the two come out")
print("close. A much larger gap in favour of near-data would be the signature of")
print("prior/posterior mismatch (section 3 checks for it directly); here they are")
print("close because the aggregate posterior roughly matches the prior. Report all")
print("three columns: high validity with low uniqueness is a model repeating itself,")
print("and novelty means nothing until you have stated the reference set.")
print("\nsome molecules sampled from the prior:")
for s in [s for s in from_prior if Chem.MolFromSmiles(s)][:5]:
    print(f"  {s}")

# A floor guard, not a quoted number: if this trips, the decoder has lost teacher
# forcing or collapsed, and generation is back to producing nothing valid.
assert results["from the prior"]["validity"] > 0.10, (
    "prior samples are almost all invalid; check the autoregressive decoder")

## Manuscript figure


- **This is Figure 10 of the manuscript** (`property_confound`): a latent space that looks organized by a property and is organized by molecular size.


This cell produces `property_confound`, the article's Section 10 figure and the argument of
section 4 above made on data instead of on a latent space: a PCA embedding of QM7
composition-and-shape descriptors colored by atomization energy, which looks beautifully
organized (panel a), beside the regression showing that atomization energy is very nearly a
function of atom count alone (panel b). It replaces an earlier VAE latent-space figure; the claim
is the same and the confound is quantified instead of asserted. It is written as **both a PDF and
a PNG**.

The published version carries no plot titles, because that text belongs in the captions. The
panel letters are a different thing and they stay: (a) and (b) say which panel you are looking
at, they do not state the figure's claim. The last line calls `set_style()` again to put the
screen defaults back, so later cells keep their titles.

In [ ]:
# --- Manuscript figure: property_confound (Section 10).
#
# The figure code lives here rather than in a script, so that the notebook a
# reader follows and the figure the article prints cannot drift apart.

import os

from ase.io import read
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler


def savefig(fig, stem, outdir=None):
    """Write a figure as both a PNG and a PDF to a local ``figures/`` directory.

    Set the FIGDIR environment variable, or pass outdir, to write somewhere else.
    """
    outdir = Path(outdir or os.environ.get("FIGDIR") or "figures")
    outdir.mkdir(parents=True, exist_ok=True)
    for fmt in ("png", "pdf"):
        fig.savefig(outdir / f"{stem}.{fmt}")
    return outdir


# scripts/make_manuscript_figures.py sets FIGDIR to collect the figures; without
# it, savefig falls back to the local figures/ directory.
FIGDIR = os.environ.get("FIGDIR")

# Figures are sized for a two-column LaTeX layout, so the type has to be
# smaller than a screen-oriented default. 7.5 pt on this canvas is 7.5 pt on
# the page, because each figure is authored at exactly the width LaTeX
# includes it at and so is never rescaled.
mpl.rcParams.update(
    {
        "figure.dpi": 200,
        "font.size": 7.5,
        "axes.labelsize": 7.5,
        "axes.titlesize": 8,
        "legend.fontsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.7,
        "ytick.major.width": 0.7,
        "xtick.major.size": 2.8,
        "ytick.major.size": 2.8,
        "lines.linewidth": 1.3,
    }
)

# Canonical figure widths, in inches, measured from the compiled document:
#   \linewidth = 250.95 pt = 3.47 in  (one column of the two-column layout)
#   \textwidth = 520.40 pt = 7.20 in  (both columns, i.e. a figure* float)
COL_W = 3.47
FULL_W = 7.20

# Panel labels: one size and offset for the whole article, so that (a) in one
# figure looks like (a) in another.
PANEL_SIZE = 8.5
PANEL_X, PANEL_Y = -0.09, 1.01


def scatter_cmap():
    """A two-color map between two distant palette colors.

    A single-hue ramp left the atomization-energy gradient in panel (a) too
    subtle to read. Two strongly contrasting endpoints make the apparent
    organization the panel is about visible at a glance, and they stay
    distinguishable in grayscale because their luminances differ.
    """
    return mpl.colors.LinearSegmentedColormap.from_list(
        "manuscript_binary", [TEAL, PLUM])


def composition_of(atoms):
    """How many H, C, N, O and S atoms the molecule has."""
    symbols = atoms.get_chemical_symbols()
    return np.array([symbols.count(element) for element in ("H", "C", "N", "O", "S")],
                    dtype=float)


def shape_of(atoms):
    """Radius of gyration, the second-moment eigenvalues, and a distance histogram."""
    pos = atoms.get_positions()
    centered = pos - pos.mean(0)
    rg = np.sqrt((centered**2).sum(1).mean())
    moments = np.linalg.eigvalsh(centered.T @ centered)
    d = np.linalg.norm(pos[:, None] - pos[None, :], axis=-1)
    iu = np.triu_indices(len(atoms), 1)
    hist, _ = np.histogram(d[iu], bins=12, range=(0, 12), density=True)
    return np.concatenate([[rg], moments, hist])


# QM7 read with ASE, then a fixed random 1500 of it: enough to make the point
# without turning panel (a) into a solid block of ink.
fig_rng = np.random.default_rng(SEED)
qm7 = read(str(DATA / "qm7.xyz"), index=":")
qm7 = [qm7[i] for i in fig_rng.choice(len(qm7), size=1500, replace=False)]

comp = np.array([composition_of(a) for a in qm7])
shape = np.array([shape_of(a) for a in qm7])
energies = np.array([a.info["atomization_energy"] for a in qm7])
n_atoms = np.array([len(a) for a in qm7])

features = np.concatenate([comp, shape], axis=1)
features = features[:, features.std(axis=0) > 0]
embedding = PCA(n_components=2, random_state=SEED).fit_transform(
    StandardScaler().fit_transform(features)
)

# The three numbers the figure reports: how strongly the embedding lines up
# with the color, and how much of that color a linear fit on atom count, and
# then on composition, already explains.
r_embed = max(abs(np.corrcoef(embedding[:, k], energies)[0, 1]) for k in (0, 1))
count_fit = LinearRegression().fit(n_atoms.reshape(-1, 1), energies)
r2_count = count_fit.score(n_atoms.reshape(-1, 1), energies)
r2_comp = LinearRegression().fit(comp, energies).score(comp, energies)

fig, axes = plt.subplots(1, 2, figsize=(FULL_W * 0.70, 2.05))
fig.subplots_adjust(wspace=0.42)

sc = axes[0].scatter(
    embedding[:, 0], embedding[:, 1], c=energies, cmap=scatter_cmap(),
    s=4, alpha=0.75, linewidths=0,
)
axes[0].set_xlabel("PC 1")
axes[0].set_ylabel("PC 2")
axes[0].set_xticks([])
axes[0].set_yticks([])
# Horizontal, below the panel: a vertical bar here collided with the
# neighboring panel's tick labels.
cb = fig.colorbar(sc, ax=axes[0], orientation="horizontal", fraction=0.055, pad=0.16)
cb.set_label("atomization energy (kcal mol$^{-1}$)", fontsize=6.8)
cb.ax.tick_params(labelsize=6.2)
cb.outline.set_edgecolor(SLATE)

axes[1].scatter(n_atoms, energies, s=4, alpha=0.4, linewidths=0, color=PURPLE)
grid = np.linspace(n_atoms.min(), n_atoms.max(), 50).reshape(-1, 1)
axes[1].plot(grid, count_fit.predict(grid), color=SLATE, lw=1.3, marker="none")
axes[1].set_xlabel("number of atoms")
axes[1].set_ylabel("atomization energy (kcal mol$^{-1}$)", fontsize=7)
axes[1].text(
    0.97, 0.95, f"$R^2 = {r2_count:.2f}$ from atom count alone\n"
                f"$R^2 = {r2_comp:.3f}$ from composition",
    transform=axes[1].transAxes, fontsize=6.6, color=SLATE,
    ha="right", va="top", linespacing=1.4,
)
# Panel letters are ax.text, not titles: they say which panel this is, they do
# not state the figure's claim.
for ax, label in zip(axes, ["(a)", "(b)"]):
    ax.text(PANEL_X, PANEL_Y, label, transform=ax.transAxes, fontsize=PANEL_SIZE,
            fontweight="bold", va="bottom", ha="left", color=SLATE)

outdir = savefig(fig, "property_confound", FIGDIR)
print(f"wrote property_confound.png/.pdf to {outdir}")
print(f"   embedding axis vs atomization energy : |r| = {r_embed:.3f}")
print(f"   atomization energy from atom count   : R^2 = {r2_count:.4f}")
print(f"   atomization energy from composition  : R^2 = {r2_comp:.4f}")
plt.show()

# Leave the notebook as we found it, so later cells keep their titles and
# their screen-sized type.
set_style()

In [ ]:
# --- Numbers the article quotes -------------------------------------------
# These assertions exist because continuous integration proves the notebook
# *runs*; it does not prove it still says what the article says it says. A
# library default changes, a result shifts, CI stays green, and the article is
# quietly wrong. Tolerance bands, not equality: catch a change that matters,
# not floating-point noise. See Section 11.3 on dependency rot.

print(f"R^2, energy from atom count        {r2_count:8.2f}   article: 0.95")
print(f"R^2, energy from composition       {r2_comp:8.3f}   article: 0.992")
print(f"|r|, embedding axis vs energy      {r_embed:8.2f}   article: 0.71")
assert r2_count > 0.90, f"atom-count R^2 drifted: {r2_count:.3f}"
assert r2_comp > 0.97, f"composition R^2 drifted: {r2_comp:.3f}"
assert r_embed > 0.55, f"embedding correlation drifted: {r_embed:.3f}"
print("\nall within tolerance of the values printed in the article")

### Exercise

Re-train with the reparameterization deliberately broken: change `eps = torch.randn_like(logvar)`
to `eps = 1e-2 * torch.randn_like(logvar)`. What happens to the aggregate posterior statistics in
section 3, and to the gap between the two sampling schemes in section 5?

In [ ]:
# YOUR CODE HERE